<a href="https://colab.research.google.com/github/HimnshuKumar7/Grp_CH4124-notebook-/blob/main/Decision_tree_using_Ginni_impurity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

In [2]:
class Node:
  def __init__(self, feature = None, threshold = None, left= None, right = None, value =None):
    self.feature = feature
    self.threshold = threshold
    self.left =left
    self.right = right

    self.value = value


In [3]:
class DecisionTree:
  def __init__(self, max_depth=5, min_samples_split= 10, min_samples_leaf = 5):
    self.max_depth = max_depth
    self.min_samples_split = min_samples_split
    self.max_samples_leaf = min_samples_leaf

  def gini(self, y):
    if len(y) == 0:
      return 0

    classes, counts = np.unique(y, return_counts = True)
    probabilities = counts / len(y)
    return 1 - np.sum(probabilities ** 2)

  def ginni_split(self, y_left, y_right):
    n = len(y_left) + len(y_right)
    weight_left = len(y_left) /n
    weight_right = len(y_right) / n
    return (weight_left * self.gini(y_left) + weight_right * self.gini(y_right))

  def best_split(self, X, y):
    n_samples, n_features = X.shape
    best_gini = float('inf')

    best_feature = None
    best_threshold = None

    for feature in range(n_features):
      values = X[:, feature]
      thresholds = np.unique(values)

      for threshold in thresholds:
        left_mask = values <= threshold
        right_mask = values > threshold

        if np.sum(left_mask) == 0:
          continue

        if np.sum(right_mask) == 0:
          continue

        y_left = y[left_mask]
        y_right = y[right_mask]

        gini = self.gini_split(y_left, y_right)
        if gini < best_gini:
          best_gini = gini
          best_feature = feature
          best_threshold = threshold

    return best_feature, best_threshold, best_gini

  def majority_class(self, y):
    classes, counts = np.unique(y, return_counts = True)
    return classes[np.argmax(counts)]

  def build_tree(self, X,y,depth = 0):
    n_samples = len(y)
    if len(np.unique(y)) == 1:
      return Node(value = y[0])

    if depth >= self.max_depth:
      return Node(value = self.majority_class(y))

    if n_samples < self.min_samples_split:
      return Node(value = self.majority_class(y))

    feature, threshold, gini = self.best_split(y)

    if feature is None:
      return Node(value = self.majority_class(y))

    left_mask = X[:,feature] <= threshold
    right_mask = X[:,feature] > threshold

    X_left = X[left_mask]
    y_left = y[left_mask]

    X_right  = X[right_mask]
    y_right = y[right_mask]

    left_child = self.build_tree(X_left, y_left, depth+1)
    right_child = self.build_tree(X_right, y_right, depth +1)

    return Node(feature = feature, threshold = threshold, left=left_child, right = right_child)

  def fit(self, X,y):
    self.root =self.build_tree(X,y,depth = 0)


  def predict_one(self, x,node):
    if node.value is not None:
      return node.value

    if x[node.feature] <= node.threshold:
      return self.predict_one(x,node.left)
    else:
      return self.predict_one(x,node.right)

  def predict(self, X):
    predictions = []
    for x in X:
      prediction = self.predict_one(x, self.root)
      predictions.append(prediction)

    return np.array(predictions)






3 important parameters to hypertune
1. max_depth
2. min_samples_split
3. min_samples_leaf


                Training Data
                     ↓
              Calculate Gini
                     ↓
          Try every feature
                     ↓
         Try every threshold
                     ↓
       Calculate weighted Gini
                     ↓
             Choose minimum
                     ↓
                 Split
                /     \
               /       \
          Left node   Right node
             ↓            ↓
        Calculate      Calculate
          Gini           Gini
             ↓            ↓
          Split          Split
             ↓            ↓
                  ...
                     ↓
                Leaf node
                     ↓
              Majority class